# 🏆 Torneio de Políticas v1 — `ptq-policy-tournament-v1`

O experimento decisivo **pré-registrado** de `docs/ALGORITMOS-DECISAO.md`:

- **A0** — heurística v0 (genoma 0). **A1** — CMA-ES sobre o genoma de 20 genes,
  candidato como empresa **focal** invadindo campos congelados **F0** (rivais v0) e
  **F1** (liga: 50% v0, 25% mutantes, 25% campeões congelados). **A2** — IPPO focal:
  documentado, **não implementado** neste notebook (ver a seção própria).
- **Orçamento equalizado**: 8.192 episódios de treino por método aprendiz;
  seleção ≤ 512 episódios. Seeds separadas: treino 1101–1164, validação
  2201–2232, teste **9901–9932 travadas** (a avaliação final exige o modo
  final explícito — célula 14).
- **Métrica**: ΔS pareado vs A0 (percentis de caixa 50% + elo 30% +
  supercomputadores 20%), IC95 por bootstrap; portões de diversidade/narrativa.
- **Vereditos**: PROVA / MATA / PLANO_B / INCONCLUSIVO, pelos critérios
  numéricos do protocolo, impressos pelo próprio notebook.

Regra do fundador: treino de verdade roda AQUI, no Kaggle — nunca na VM
de desenvolvimento. Localmente este notebook só roda em modo FUMAÇA
(`PTQ_TORNEIO_SMOKE=1`), com orçamento reduzido e **sem tocar as seeds
de teste**.

In [ ]:
# 1) Fonte do pacote: dataset privado com o fonte do repo (bundle novo:
#    src4 — ver scripts/kaggle_push_kernel.py; o pacote e puro Python,
#    NAO precisa de pip install. Basta src/ no path.)
#    Fora do Kaggle (validacao local), cai para o src/ do repo.
import glob, os, sys, zipfile

cands = glob.glob('/kaggle/input/**/mundo*.zip', recursive=True) + \
        glob.glob('/kaggle/input/*/src/mundo/__init__.py')
if cands and cands[0].endswith('.zip'):
    zipfile.ZipFile(cands[0]).extractall('/kaggle/working/pkg')
    src_root = '/kaggle/working/pkg'
elif cands:
    src_root = cands[0].split('/src/')[0]
elif os.path.isfile('src/mundo/__init__.py'):
    src_root = os.getcwd()                      # validacao local (fumaca)
elif os.path.isfile('../src/mundo/__init__.py'):
    src_root = os.path.abspath('..')
else:
    raise SystemExit('Fonte do pacote nao encontrada: anexe o dataset '
                     'com o fonte do repo (mundo-autonomo-src4) a este kernel.')
print('pacote em', src_root)
sys.path.insert(0, os.path.join(src_root, 'src'))
import mundo
print('mundo', mundo.__version__, 'importado de', src_root)

## Configuração — o orçamento é o protocolo

No modo oficial, os números abaixo SÃO o orçamento pré-registrado:
2 corridas CMA-ES × λ=16 × 32 gerações × 8 blocos = **8.192 episódios**.
O modo fumaça (validação local) reduz tudo e troca a avaliação final
por uma **simulada sobre seeds de validação**.

In [ ]:
# 2) Configuracao do torneio
import time

SMOKE = os.environ.get('PTQ_TORNEIO_SMOKE') == '1'
PROCESSES = int(os.environ.get('PTQ_PROCESSES', '4'))

if SMOKE:
    CFG = dict(runs=1, popsize=4, generations=2, blocks_per_candidate=2,
               ticks=24, sigma=0.5, cma_seeds=(5,))
    BUDGET_TRAIN = 4 * 2 * 2
else:
    CFG = dict(runs=2, popsize=16, generations=32, blocks_per_candidate=8,
               ticks=120, sigma=0.5, cma_seeds=(11, 22))
    BUDGET_TRAIN = 8192
print('SMOKE' if SMOKE else 'OFICIAL', '| config:', CFG,
      '| processos:', PROCESSES)

In [ ]:
# 3) Sanidade: genoma 0 == parametros v0, e o placar de referencia existe
from mundo.engine import HeuristicPolicy
from mundo.genome import GENOME0, genome_to_params, policy_id

assert genome_to_params(GENOME0) == HeuristicPolicy.PARAMS0
assert policy_id(GENOME0) == 'v0'
print('genoma 0 OK:', policy_id(GENOME0), '| genes:', len(GENOME0))

## A1 — CMA-ES em liga

Cada candidato é avaliado como empresa focal em blocos F0/F1 alternados
(seeds de treino, girando por geração). O campeão de cada geração congela
e entra na liga F1 das seguintes. O arquivo QD mínimo guarda elites por
célula de comportamento (elo × supercomputadores × faixa de caixa).

In [ ]:
# 4) Treino A1: corridas de CMA-ES com contabilidade de episodios
from mundo import tournament as T
from mundo.genome import GENOME0, genome_hash, policy_id

archive = T.BehaviorArchive()
run_results, ep_train = [], 0
t0 = time.time()
for r in range(CFG['runs']):
    res = T.run_cma(popsize=CFG['popsize'], generations=CFG['generations'],
                    blocks_per_candidate=CFG['blocks_per_candidate'],
                    sigma=CFG['sigma'], seed=CFG['cma_seeds'][r],
                    processes=PROCESSES, ticks=CFG['ticks'], archive=archive)
    run_results.append(res)
    ep_train += res['episodes']
    print(f"corrida {r}: campeao {policy_id(res['champion'])} "
          f"fitness {res['champion_fitness']:.3f} | "
          f"episodios {res['episodes']} | {time.time() - t0:.0f}s")
assert ep_train <= BUDGET_TRAIN, (ep_train, BUDGET_TRAIN)
print(f'treino: {ep_train}/{BUDGET_TRAIN} episodios | '
      f'arquivo QD: {len(archive.cells)} celulas')

## Seleção — validação, com teto duro de 512 episódios

Os candidatos (campeões das corridas + elites do arquivo QD) são
reavaliados nas seeds de **validação**; o campeão final é o de maior
média de S. Nenhuma seed de teste é tocada aqui.

In [ ]:
# 5) Selecao do campeao A1 nas seeds de validacao
seen, candidates = set(), []
for res in run_results:
    candidates.append(res['champion'])
for g in archive.genomes():
    if policy_id(g) not in {policy_id(c) for c in candidates}:
        candidates.append(g)
candidates = candidates[:4]              # 4 x 32 = 128 <= 512 episodios
sel = T.select_champion(candidates, processes=PROCESSES, ticks=CFG['ticks'])
CHAMPION = sel['champion']
print('tabela de selecao (media de S na validacao):')
for pid, mean_s in sel['table']:
    print(f'  {pid}: {mean_s:.3f}')
print(f"campeao A1: {policy_id(CHAMPION)} "
      f"(hash {genome_hash(CHAMPION)}, S={sel['champion_mean_S']:.3f}, "
      f"{sel['episodes']} episodios de selecao)")

## Portões de produto — a ecologia do campeão

Mundos inteiros com TODAS as empresas usando o campeão, contra mundos
inteiros v0 (mesmas seeds de validação): eventos narráveis e diversidade
comportamental do campeão precisam ser ≥ 80% dos do A0, e nenhuma célula
comportamental pode passar de 50% das empresas.

In [ ]:
# 6) Ecologia e portoes
eco_champ = T.ecology_metrics(CHAMPION, processes=PROCESSES,
                              ticks=CFG['ticks'])
eco_base = T.ecology_metrics(GENOME0, processes=PROCESSES,
                             ticks=CFG['ticks'])
GATES = T.gates_ok(eco_champ, eco_base)
print('ecologia campeao:', {k: round(v, 3) if isinstance(v, float) else v
                            for k, v in eco_champ.items()})
print('ecologia A0     :', {k: round(v, 3) if isinstance(v, float) else v
                            for k, v in eco_base.items()})
print('portoes de produto:', 'OK' if GATES else 'REPROVADO')

## A2 — IPPO focal: documentado, NÃO implementado neste notebook

**Por que não entrou:** o braço A2 exige a camada `Observation` (o
contrato real hoje é `Policy.decide(engine, comp)` — uma política RL
não pode ler o motor inteiro sem quebrar o fog-of-war por construção)
e um harness de treino PPO com máscaras de ação e recompensa de
patrimônio líquido. Sem essa camada, um "IPPO" seria teatro: treinaria
sobre informação ilegal ou sobre um problema diferente do registrado.

**Contrato quando for implementado** (protocolo, seção do plano B):
empresa focal com parameter sharing + embedding de arquétipo, rivais
congelados dos mesmos campos F0/F1, MESMO orçamento de 8.192 episódios,
mesma métrica ΔS — e o veredito só vira PLANO_B se o A2 vencer o A1 por
≥ +0,08 com IC95 > 0. Enquanto o A2 não existe, PLANO_B é inalcançável
e o veredito sai entre PROVA / MATA / INCONCLUSIVO.

In [ ]:
# 7) A2: ausente por construcao (ver a secao acima)
A2_CHAMPION = None
print('A2 (IPPO focal): NAO IMPLEMENTADO neste notebook — '
      'PLANO_B inalcancavel; veredito entre PROVA/MATA/INCONCLUSIVO.')

## Avaliação final — as seeds travadas

Os 64 blocos do teste: por seed de teste, um bloco **F0 sem choque** e
um bloco **F2 com os choques pré-registrados** (descoberta aos 30,
choque de preço do lítio aos 60, sanção ao cobre aos 90).

- **Modo oficial**: exige `RUN_FINAL = True` — deliberado, na célula.
  Sem isso, `evaluate_test` levanta `TestSeedLock` e nada roda.
- **Modo fumaça**: a "final" é simulada sobre seeds de VALIDAÇÃO com
  os mesmos campos/choques — prova o encanamento sem abrir a trava.

In [ ]:
# 8) Avaliacao final pareada (campeao A1 x A0, mesmos blocos)
RUN_FINAL = True   # <-- o lancamento oficial poe True, deliberadamente

if SMOKE:
    sim_blocks = []
    for s in T.VAL_SEEDS[:8]:
        sim_blocks.append((s, 'F0', False))
        sim_blocks.append((s, 'F2', True))
    champ_blocks = T.evaluate_blocks(CHAMPION, sim_blocks,
                                     processes=PROCESSES, ticks=CFG['ticks'])
    base_blocks = T.evaluate_blocks(GENOME0, sim_blocks,
                                    processes=PROCESSES, ticks=CFG['ticks'])
    print('FINAL SIMULADA (fumaca, seeds de validacao):',
          len(sim_blocks), 'blocos')
elif RUN_FINAL:
    champ_blocks = T.evaluate_test(CHAMPION, final=True,
                                   processes=PROCESSES, ticks=CFG['ticks'])
    base_blocks = T.evaluate_test(GENOME0, final=True,
                                  processes=PROCESSES, ticks=CFG['ticks'])
    print('FINAL OFICIAL (seeds de teste):', len(champ_blocks), 'blocos')
else:
    champ_blocks = base_blocks = None
    print('RUN_FINAL=False: avaliacao final NAO executada. '
          'Para o veredito oficial, ponha RUN_FINAL=True e rode de novo.')

In [ ]:
# 9) Veredito — criterios numericos pre-registrados, impressos aqui
import json

import numpy as np

if champ_blocks is None:
    VERDICT = 'NAO AVALIADO'
    print('VEREDITO: NAO AVALIADO (avaliacao final nao executada)')
else:
    deltas = T.paired_deltas(champ_blocks, base_blocks)
    d_f2 = [m['S'] - b['S'] for m, b in zip(champ_blocks, base_blocks)
            if m['field'] == 'F2']
    d_shock = [m['S'] - b['S'] for m, b in zip(champ_blocks, base_blocks)
               if m['shocks']]
    DELTA_S = float(np.mean(deltas))
    CI = T.bootstrap_ci(deltas)
    DELTA_F2 = float(np.mean(d_f2)) if d_f2 else 0.0
    DELTA_SHOCK = float(np.mean(d_shock)) if d_shock else 0.0
    VERDICT = T.verdict(DELTA_S, CI, DELTA_F2, DELTA_SHOCK, GATES)
    print(f'DeltaS (A1 - A0) = {DELTA_S:+.3f}  IC95 [{CI[0]:+.3f}, '
          f'{CI[1]:+.3f}]  n={len(deltas)} blocos pareados')
    print(f'DeltaS F2 = {DELTA_F2:+.3f} | DeltaS choques = '
          f'{DELTA_SHOCK:+.3f} | portoes: {"OK" if GATES else "FALHOU"}')
    print(f'VEREDITO: {VERDICT}' + (' (SIMULADO — fumaca)'
                                   if SMOKE else ''))

resultado = {
    'verdict': VERDICT,
    'smoke': SMOKE,
    'champion_policy_id': policy_id(CHAMPION),
    'champion_genome_hash': genome_hash(CHAMPION),
    'champion_genome': list(CHAMPION),
    'champion_params': genome_to_params(CHAMPION),
    'episodes_train': ep_train,
    'episodes_selection': sel['episodes'],
    'selection_table': sel['table'],
    'ecology_champion': eco_champ,
    'ecology_a0': eco_base,
    'gates_ok': GATES,
    'a2': 'nao implementado (ver secao A2 do notebook)',
}
if champ_blocks is not None:
    resultado.update({'delta_s': DELTA_S, 'ci95': list(CI),
                      'delta_f2': DELTA_F2, 'delta_shock': DELTA_SHOCK})
out_dir = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
out_path = os.path.join(out_dir, 'genomas_campeoes.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(resultado, f, ensure_ascii=False, indent=1)
print('salvo:', out_path)